In [2]:
# Cell 1: Import Thư viện
import sys
import os
import pandas as pd
import yfinance as yf
import joblib
import warnings

warnings.filterwarnings('ignore')
sys.path.append(os.path.abspath(".."))
from utils.feature_engineer import TechnicalFeatures

# Cell 2: Tải Mô hình AI đã huấn luyện
model_path = "../data/processed/xgboost_model.pkl"
if not os.path.exists(model_path):
    raise FileNotFoundError(f"Không tìm thấy file mô hình tại {model_path}. Hãy chạy Notebook huấn luyện trước!")

model = joblib.load(model_path)
expected_features = list(model.feature_names_in_)
print(f"✅ Đã nạp Mô hình XGBoost thành công. Yêu cầu {len(expected_features)} đặc trưng.")

# Cell 3: Tải dữ liệu thị trường mới nhất
# ĐỔI TỪ "GAS" THÀNH "GAS.VN" ĐỂ LẤY ĐÚNG CỔ PHIẾU VIỆT NAM
ticker = "GAS.VN" 
print(f"Đang tải dữ liệu thời gian thực cho {ticker}...")

# Tải 200 ngày gần nhất để đảm bảo đủ dữ liệu tính các chỉ báo chu kỳ dài (SMA50, SMA200...)
df_live = yf.download(ticker, period="200d", interval="1d", progress=False)

if df_live.empty:
    raise ValueError(f"Không thể tải dữ liệu cho mã {ticker}. Hãy kiểm tra lại kết nối mạng hoặc Ticker!")

# Xử lý triệt để MultiIndex của yfinance bản mới
if isinstance(df_live.columns, pd.MultiIndex):
    df_live.columns = df_live.columns.get_level_values(0)

# Cell 4: Xử lý dữ liệu (Feature Engineering)
te_live = TechnicalFeatures(df_live)
df_live_features = te_live.generate_all_features()

# Lọc bỏ các dòng NaN xuất hiện do việc tính toán các chỉ báo kỹ thuật (Lag/SMA)
df_clean_live = df_live_features.dropna(subset=expected_features)

if df_clean_live.empty:
    raise ValueError("Dữ liệu sau khi tính đặc trưng bị rỗng hoàn toàn (do không đủ số lượng phiên giao dịch).")

# Lấy dòng dữ liệu MỚI NHẤT đã sạch NaN
today_data = df_clean_live.iloc[[-1]].copy()

latest_date = pd.to_datetime(today_data.index[0]).strftime("%Y-%m-%d")
latest_close_price = float(today_data['Close'].values[0])

# Trích xuất đúng các cột đặc trưng mà mô hình yêu cầu
X_live = today_data[expected_features]

# Cell 5: AI Đưa ra Dự đoán (Inference)
prediction = model.predict(X_live)[0]
probability = model.predict_proba(X_live)[0]

# In Báo cáo Tín hiệu
print("\n" + "★"*50)
print(f" TÍN HIỆU GIAO DỊCH AI - MÃ: {ticker} ")
print(f" Ngày cập nhật phiên gần nhất: {latest_date}")
print(f" Giá đóng cửa: {latest_close_price:,.0f} VNĐ")
print("★"*50)

if prediction == 1:
    print(">> KHUYẾN NGHỊ: MUA / NẮM GIỮ (BULLISH) 🟢")
    print(f">> Độ tự tin (Xác suất tăng): {probability[1]*100:.2f}%")
else:
    print(">> KHUYẾN NGHỊ: BÁN / ĐỨNG NGOÀI (BEARISH) 🔴")
    print(f">> Độ tự tin (Xác suất giảm): {probability[0]*100:.2f}%")
    
print("★"*50)

✅ Đã nạp Mô hình XGBoost thành công. Yêu cầu 10 đặc trưng.
Đang tải dữ liệu thời gian thực cho GAS.VN...


2026-07-31 11:54:40,195 [INFO] Bắt đầu tính toán Technical Indicators & Macro Features...
2026-07-31 11:54:40,829 [INFO] Hoàn tất. Tổng số features (bao gồm vĩ mô): 21
2026-07-31 11:54:40,830 [INFO] Đã loại bỏ 49 dòng NaN ở đầu chuỗi dữ liệu.



★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★
 TÍN HIỆU GIAO DỊCH AI - MÃ: GAS.VN 
 Ngày cập nhật phiên gần nhất: 2026-07-31
 Giá đóng cửa: 68,700 VNĐ
★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★
>> KHUYẾN NGHỊ: BÁN / ĐỨNG NGOÀI (BEARISH) 🔴
>> Độ tự tin (Xác suất giảm): 63.28%
★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★
